# CityLens + GDP baseline in Colab

Runs everything in the cloud: clone repo, **dataset in Google Drive** (download once, reuse forever), install deps, run GDP evaluation with Gemini.

---

## What to do now

1. **Upload this notebook to Colab** (File → Upload notebook, or open from GitHub).
2. **Runtime → Run all** (or run cells in order).
3. **First run:** Section 3 will ask you to **mount Google Drive** (click the link, allow access). Then it downloads ~10 GB and extracts to **My Drive → GeoBench → CityLens-Data**. Eval and metrics run; results are also saved in that same Drive folder.
4. **Next runs (new session):** Run all again. Section 3 mounts Drive and **reuses** existing data (no re-download). You get your data back automatically.

**Drive check:** After section 3, open [drive.google.com](https://drive.google.com) → **My Drive → GeoBench → CityLens-Data**. You should see the dataset folders (and after eval, a **Results** folder). That’s your persisted data.

**Before running:** Stop any local download on your Mac (Ctrl+C in the terminal).

## 1. Clone CityLens repo

In [ ]:
!git clone --depth 1 https://github.com/tsinghua-fib-lab/CityLens.git
%cd CityLens

Cloning into 'CityLens'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 84 (delta 26), reused 36 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 2.03 MiB | 8.60 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/CityLens


## 2. Add path config and patch eval code for Colab

In [ ]:
import os

# Write into CityLens/ so eval finds it when run from /content/CityLens
target_dir = "CityLens" if os.path.isdir("CityLens") else "."
path_config_file = os.path.join(target_dir, "path_config.py")
PATH_CONFIG = '''"""
Path config for Colab. Data root = /content/CityLens-Data
"""
import os

DATA_ROOT = os.environ.get("CITYLENS_DATA_ROOT", "/content/CityLens-Data")

def benchmark_path(task_name: str, city: str) -> str:
    return os.path.join(DATA_ROOT, "Benchmark", f"{task_name}_{city}.json")

def results_path(task_name: str, city: str, model_name: str, prompt_type: str) -> str:
    safe = model_name.replace("/", "_")
    return os.path.join(DATA_ROOT, "Results", f"{task_name}_{city}_{safe}_{prompt_type}.json")

def summary_csv_path() -> str:
    return os.path.join(DATA_ROOT, "Results", "summary.csv")

def url_file_path() -> str:
    return os.path.join(DATA_ROOT, "Benchmark", "image_urls.csv")
'''

with open(path_config_file, "w") as f:
    f.write(PATH_CONFIG)
print("Wrote", path_config_file)

Wrote ./path_config.py


: 

: 

**Run this next:** Patch config.py so utils.py does not raise ImportError (adds SILICONFLOW_APIKEY, DEEPINFRA_APIKEY).

In [ ]:
# Add missing config vars so utils.py can import (not needed for Gemini)
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "config.py"
with open(path, "r") as f: text = f.read()
if "SILICONFLOW_APIKEY" not in text:
    text += "\n# Optional API keys (not needed for Gemini)\nSILICONFLOW_APIKEY = \"\"\nDEEPINFRA_APIKEY = \"\"\n"
    with open(path, "w") as f: f.write(text)
    print("Patched config.py with SILICONFLOW_APIKEY, DEEPINFRA_APIKEY")
else:
    print("config.py already has API keys.")

Patched config.py with SILICONFLOW_APIKEY, DEEPINFRA_APIKEY


: 

: 

In [ ]:
# Patch global_indicator.py to use path_config
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "evaluate/global/global_indicator.py"
with open(path, "r") as f:
    text = f.read()

if "path_config" not in text:
    text = text.replace(
        "from config import STV_NUM, GLOBAL_TASK_NUM\nrandom.seed(42)",
        "from config import STV_NUM, GLOBAL_TASK_NUM\n\ntry:\n    from path_config import benchmark_path, results_path, url_file_path\n    _USE_PATH_CONFIG = True\nexcept ImportError:\n    _USE_PATH_CONFIG = False\n\nrandom.seed(42)"
    )
    text = text.replace(
        '    url_file = ""',
        '    url_file = url_file_path() if _USE_PATH_CONFIG else ""'
    )
    text = text.replace(
        "if city == \"all\":\n        task_path = f\"\"\n    else:\n        task_path = f'\''\n    \n    model_name_full = model_name.replace(\"/\", \"_\")\n    response_path = f'\''",
        "model_name_full = model_name.replace(\"/\", \"_\")\n    if _USE_PATH_CONFIG:\n        task_path = benchmark_path(task_name, city)\n        response_path = results_path(task_name, city, model_name, prompt_type)\n    else:\n        if city == \"all\":\n            task_path = f\"\"\n        else:\n            task_path = f'\''\n        response_path = f'\''"
    )
    import re
    # Replace eval_task path block (allow flexible whitespace)
    pattern = r"(def eval_task\(city, model_name, num_process, prompt_type, task_name\):)\s+if city == \"all\":\s+task_path = f\"\"\s+else:\s+task_path = f'\''\s+model_name_full = model_name\.replace\(\"/\", \"_\"\)\s+response_path = f'\''"
    repl = r"\1\n    model_name_full = model_name.replace(\"/\", \"_\")\n    if _USE_PATH_CONFIG:\n        task_path = benchmark_path(task_name, city)\n        response_path = results_path(task_name, city, model_name, prompt_type)\n    else:\n        if city == \"all\":\n            task_path = f\"\"\n        else:\n            task_path = f'\''\n        response_path = f'\''"
    text = re.sub(pattern, repl, text, count=1)
    with open(path, "w") as f:
        f.write(text)
    print("Patched global_indicator.py")
else:
    print("global_indicator.py already patched")

# Add fallback so task_path/response_path set from env when path_config not loaded
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "evaluate/global/global_indicator.py"
with open(path, "r") as f: t = f.read()
needle = "    output_dir = os.path.dirname(response_path)"
fallback_insert = "    # Fallback when path_config not loaded (e.g. Colab)\n    if not task_path or not response_path:\n        data_root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"/content/CityLens-Data\")\n        task_path = os.path.join(data_root, \"Benchmark\", f\"{task_name}_{city}.json\")\n        response_path = os.path.join(data_root, \"Results\", f\"{task_name}_{city}_{model_name_full}_{prompt_type}.json\")\n    " + needle
if "if not task_path or not response_path" in t:
    print("eval_task fallback already present.")
elif needle in t:
    before, _, after = t.partition(needle)
    t = before + fallback_insert + after
    t = t.replace("    if not os.path.exists(output_dir):", "    if output_dir and not os.path.exists(output_dir):")
    with open(path, "w") as f: f.write(t)
    print("Added eval_task fallback for Colab.")
else:
    t = t.replace("    if not os.path.exists(output_dir):", "    if output_dir and not os.path.exists(output_dir):")
    with open(path, "w") as f: f.write(t)
    print("Patched makedirs guard. Set CITYLENS_DATA_ROOT.")

Patched global_indicator.py
Added eval_task fallback for Colab.


: 

: 

In [ ]:
# Patch metrics.py to use path_config
path = "evaluate/global/metrics.py"
with open(path, "r") as f:
    text = f.read()

if "path_config" not in text:
    text = text.replace(
        "from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n",
        "from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n\ntry:\n    from path_config import results_path, summary_csv_path\n    _USE_PATH_CONFIG = True\nexcept ImportError:\n    _USE_PATH_CONFIG = False\n"
    )
    old = """    args = parser.parse_args()
    model_name_full = args.model_name.replace("/", "_")
    response_path = f''
    summary_path = f''"""
    new = """    args = parser.parse_args()
    model_name_full = args.model_name.replace("/", "_")
    if _USE_PATH_CONFIG:
        response_path = results_path(args.task_name, args.city_name, args.model_name, args.prompt_type)
        summary_path = summary_csv_path()
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
    else:
        response_path = f''
        summary_path = f''"""
    text = text.replace(old, new)
    with open(path, "w") as f:
        f.write(text)
    print("Patched metrics.py")
else:
    pass
text = open(path).read()
if "if not response_path:" not in text:
    old_block = """        summary_path = f''
    y_true, y_pred"""
    new_block = """        summary_path = f''
    if not response_path:
        data_root = os.environ.get("CITYLENS_DATA_ROOT", "")
        if data_root:
            response_path = os.path.join(data_root, "Results", f"{args.task_name}_{args.city_name}_{model_name_full}_{args.prompt_type}.json")
            summary_path = os.path.join(data_root, "Results", "summary.csv")
            os.makedirs(os.path.dirname(summary_path), exist_ok=True)
    y_true, y_pred"""
    text = text.replace(old_block, new_block)
    with open(path, "w") as f: f.write(text)
    print("Added metrics env fallback.")
else:
    print("metrics.py already patched")

Patched metrics.py
Added metrics env fallback.


: 

: 

### 2b. Gemini adapter (creates gemini_adapter.py + patches utils.py)
Run the next two code cells so the eval can use Gemini (free API).

In [ ]:
# Create evaluate/gemini_adapter.py (cloned repo does not have it)
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
os.makedirs("evaluate", exist_ok=True)
adapter = '''"""
Gemini API adapter for CityLens (Google AI Studio free tier).
Set GOOGLE_API_KEY or GEMINI_API_KEY. Get key at https://aistudio.google.com/apikey
"""
import os
import base64
import time
import requests

GEMINI_MODEL_NAMES = ["gemini-1.5-flash", "gemini-1.5-pro", "gemini-1.5-flash-8b", "gemini"]

def _session_to_gemini_parts(session):
    parts = []
    if not session or session[0].get("role") != "user": return parts
    content = session[0].get("content", [])
    if isinstance(content, str): return [content]
    for item in content:
        if item.get("type") == "text": parts.append(item["text"])
        elif item.get("type") == "image_url":
            url = (item.get("image_url") or {}).get("url")
            if url:
                try:
                    resp = requests.get(url, timeout=30); resp.raise_for_status()
                    b64 = base64.b64encode(resp.content).decode("utf-8")
                    mime = (resp.headers.get("Content-Type") or "image/jpeg").split(";")[0].strip() or "image/jpeg"
                    parts.append({"inline_data": {"mime_type": mime, "data": b64}})
                except Exception as e: print(f"Warning: could not fetch image: {e}")
    return parts

def get_response_mllm_api_gemini(session, model_name, temperature=0, max_tokens=1000, max_retries=5):
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key: print("Error: Set GOOGLE_API_KEY (aistudio.google.com/apikey)"); return None
    try: import google.generativeai as genai
    except ImportError: print("Error: pip install google-generativeai"); return None
    genai.configure(api_key=api_key)
    parts = _session_to_gemini_parts(session)
    if not parts: return None
    gemini_model = "gemini-1.5-flash" if model_name == "gemini" else model_name
    retries = 0
    while retries < max_retries:
        try:
            model = genai.GenerativeModel(gemini_model)
            resp = model.generate_content(parts, generation_config={"temperature": temperature, "max_output_tokens": max_tokens})
            if resp and resp.text: return resp.text.strip()
            return None
        except Exception as e:
            retries += 1; print(f"Error calling Gemini API: {e}. Retry {retries}/{max_retries}...")
            if retries < max_retries: time.sleep(2 ** retries)
    return None
'''
with open("evaluate/gemini_adapter.py", "w") as f: f.write(adapter)
print("Created evaluate/gemini_adapter.py")

Created evaluate/gemini_adapter.py


: 

: 

In [ ]:
# Patch utils.py to use Gemini adapter
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "evaluate/utils.py"
with open(path, "r") as f: text = f.read()
if "get_response_mllm_api_gemini" in text:
    print("utils.py already has Gemini support.")
elif not os.path.isfile("evaluate/gemini_adapter.py"):
    print("Create evaluate/gemini_adapter.py in Colab (copy from project CityLens/evaluate/gemini_adapter.py), then re-run this cell.")
else:
    text = text.replace("import json\n", "import json\ntry:\n    from evaluate.gemini_adapter import GEMINI_MODEL_NAMES, get_response_mllm_api_gemini\nexcept ImportError:\n    GEMINI_MODEL_NAMES = []\n    get_response_mllm_api_gemini = None\n\n", 1)
    text = text.replace("    retries = 0\n    if model_name in", "    retries = 0\n\n    if get_response_mllm_api_gemini and model_name in GEMINI_MODEL_NAMES:\n        return get_response_mllm_api_gemini(session, model_name, temperature, max_tokens, max_retries)\n\n    if model_name in", 1)
    with open(path, "w") as f: f.write(text)
    print("Patched utils.py for Gemini.")

Patched utils.py for Gemini.


: 

: 

## 3. Google Drive + CityLens dataset (~10 GB)

**Dataset is stored in your Google Drive** so you only download once. Each run: mount Drive → reuse existing data (no re-download). You need **~15 GB free** in Drive. The first run downloads and extracts to `My Drive/GeoBench/CityLens-Data/`.

In [ ]:
import os
import zipfile
import shutil
import urllib.request
from pathlib import Path

# 1) Mount Google Drive (prompt for auth if needed)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# 2) Dataset lives in Drive so we don't re-download every session
DRIVE_DATA = Path("/content/drive/MyDrive/GeoBench/CityLens-Data")
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

# 3) Check free space on Drive (need ~15 GB for zip + extract)
try:
    stat = shutil.disk_usage(DRIVE_DATA)
    free_gb = stat.free / (1024**3)
    print(f"Google Drive free space: {free_gb:.1f} GB")
    if free_gb < 15:
        raise SystemExit("Need at least 15 GB free in Google Drive. Free up space and re-run this cell.")
except Exception as e:
    print("Could not check Drive space:", e)

def _has_benchmark(p):
    p = Path(p)
    if (p / "Benchmark").exists() and (p / "Dataset").exists():
        return True
    return any((p / d.name / "Benchmark").exists() and (p / d.name / "Dataset").exists() for d in p.iterdir() if d.is_dir())

def _find_data_root(base):
    base = Path(base)
    if (base / "Benchmark").exists() and (base / "Dataset").exists():
        return str(base)
    for d in base.iterdir():
        if d.is_dir() and (d / "Benchmark").exists() and (d / "Dataset").exists():
            return str(d)
    return None

# 4) If data already in Drive, use it (no download)
data_root = _find_data_root(DRIVE_DATA)
if data_root:
    os.environ["CITYLENS_DATA_ROOT"] = data_root
    print("Data already in Drive at:", data_root)
    print("CITYLENS_DATA_ROOT set. Skipping download.")
else:
    # 5) Download zip to /content (faster than writing zip to Drive directly)
    REPO_ID = "Tianhui-Liu/CityLens-Data"
    FILENAME = "CityLens-Data.zip"
    ZIP_PATH = Path("/content") / FILENAME
    HF_URL = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{FILENAME}"

    def progress_hook(block_num, block_size, total_size):
        if total_size and total_size > 0:
            pct = min(100, 100 * block_num * block_size / total_size)
            mb = block_num * block_size / (1024*1024)
            total_mb = total_size / (1024*1024)
            print(f"\r  {pct:.1f}% ({mb:.1f} / {total_mb:.1f} MB)", end="", flush=True)

    print("Downloading CityLens-Data (~10 GB, 10–20 min)...")
    urllib.request.urlretrieve(HF_URL, ZIP_PATH, reporthook=progress_hook)
    print("\nExtracting to Google Drive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DRIVE_DATA)
    try:
        ZIP_PATH.unlink()
    except Exception:
        pass
    data_root = _find_data_root(DRIVE_DATA)
    if not data_root:
        raise SystemExit("Extracted but Benchmark/Dataset not found under " + str(DRIVE_DATA))
    os.environ["CITYLENS_DATA_ROOT"] = data_root
    print("Done. Data root:", data_root)

print("CITYLENS_DATA_ROOT =", os.environ.get("CITYLENS_DATA_ROOT"))

  6.8% (663.7 / 9782.8 MB)

  7.0% (683.1 / 9782.8 MB)

: 

: 

In [ ]:
# Data root is set by the cell above (from Google Drive). This just confirms.
import os
from pathlib import Path
data_root = os.environ.get("CITYLENS_DATA_ROOT")
if data_root and Path(data_root).exists():
    print("Data root OK (from Drive):", data_root)
else:
    print("CITYLENS_DATA_ROOT not set or path missing. Re-run the 'Google Drive + CityLens dataset' cell above.")

: 

: 

In [ ]:
# Verify: data is stored in Google Drive (persists across sessions)
import os
root = os.environ.get("CITYLENS_DATA_ROOT", "")
if root.startswith("/content/drive"):
    drive_path = root.replace("/content/drive/MyDrive/", "My Drive/")
    print("Saved in Drive:", drive_path)
    print("Next session: run this notebook again; section 3 will mount Drive and reuse this data (no re-download).")
else:
    print("CITYLENS_DATA_ROOT:", root or "(not set)")

: 

: 

### Verify dataset (run anytime)

Run this cell to check whether the dataset exists in Colab and where it is. If you see "NOT FOUND", re-run the **Download CityLens-Data** cell (section 3); Colab's `/content/` is wiped when the runtime restarts.

In [ ]:
import os
from pathlib import Path

print("=== Dataset verification ===\n")

# Prefer CITYLENS_DATA_ROOT (set by section 3 from Drive)
data_root = os.environ.get("CITYLENS_DATA_ROOT")
if data_root and Path(data_root).exists():
    if (Path(data_root) / "Benchmark").exists() and (Path(data_root) / "Dataset").exists():
        print("✓ Data root (from Drive):", data_root)
    else:
        print("CITYLENS_DATA_ROOT set but Benchmark/Dataset missing at:", data_root)
        data_root = None

# Else check Google Drive path
if not data_root:
    drive_base = Path("/content/drive/MyDrive/GeoBench/CityLens-Data")
    if drive_base.exists():
        if (drive_base / "Benchmark").exists() and (drive_base / "Dataset").exists():
            data_root = str(drive_base)
            print("✓ Data root (Drive):", data_root)
        else:
            for d in drive_base.iterdir():
                if d.is_dir() and (d / "Benchmark").exists() and (d / "Dataset").exists():
                    data_root = str(d)
                    print("✓ Data root (Drive):", data_root)
                    break
    if not data_root and drive_base.exists():
        print("Drive folder exists but Benchmark/Dataset not found. Subdirs:", [x.name for x in drive_base.iterdir() if x.is_dir()][:10])

# Else check /content (legacy)
if not data_root:
    base = Path("/content/CityLens-Data")
    if base.exists():
        if (base / "Benchmark").exists() and (base / "Dataset").exists():
            data_root = str(base)
        else:
            for d in base.iterdir():
                if d.is_dir() and (d / "Benchmark").exists() and (d / "Dataset").exists():
                    data_root = str(d)
                    break
    if data_root:
        print("✓ Data root (/content):", data_root)

if data_root:
    print("\nCITYLENS_DATA_ROOT =", repr(data_root))
else:
    print("\n✗ No valid data root. Run section 3 (Google Drive + CityLens dataset) first.")

: 

: 

### Debug: why "0 valid data" in metrics?

If the **Run metrics** cell prints **length of vals: 0 0** or **No valid data found**, run the cell below. It inspects the results JSON (reference/response) and shows why numbers might not be extracted. Common causes: (1) Results file empty → re-run GDP eval. (2) Wrong keys in JSON → eval wrote different format. (3) Model returned verbose text → `extract_float` looks for the first number in text; if none, that item is skipped.

In [ ]:
# Run this if metrics showed "0 valid data" – inspect results JSON
import os
import json
import re
from pathlib import Path

def _extract_float(v):
    if v is None: return None
    if isinstance(v, (float, int)): return float(v)
    if isinstance(v, str):
        s = v.replace(",", "").replace("$", "").replace("\u00a3", "").strip()
        if len(s) <= 100:
            try: return float(s)
            except ValueError: pass
        m = re.search(r"-?[\d,]+\.?\d*", v)
        if m: return float(m.group().replace(",", ""))
    return None

# Find results file (same logic as metrics cell)
data_root = os.environ.get("CITYLENS_DATA_ROOT")
if not data_root or not os.path.isdir(data_root):
    data_root = None
    for base in ["/content/drive/MyDrive/GeoBench/CityLens-Data", "/content/CityLens-Data"]:
        if os.path.isdir(base):
            for r, _, files in os.walk(base):
                if "gdp_all_gemini-1.5-flash_simple.json" in files:
                    data_root = os.path.dirname(r)  # r is Results/ dir
                    break
            if data_root is not None:
                break
resp_path = os.path.join(data_root or "", "Results", "gdp_all_gemini-1.5-flash_simple.json")
if not os.path.isfile(resp_path):
    print("Results file not found:", resp_path)
    print("Re-run the GDP eval cell and wait for it to finish.")
else:
    with open(resp_path) as f:
        data = json.load(f)
    print("Results JSON length:", len(data))
    if len(data) == 0:
        print("File is EMPTY. Re-run the GDP eval cell (Patch eval_task then run).")
    else:
        first = data[0]
        print("First item keys:", list(first.keys()))
        ref, resp = first.get("reference"), first.get("response")
        print("reference (type):", type(ref).__name__, "| value:", repr(str(ref)[:120]))
        print("response (type):", type(resp).__name__, "| value:", repr(str(resp)[:120]) if resp else None)
        print("_extract_float(reference):", _extract_float(ref))
        print("_extract_float(response):", _extract_float(resp))
        valid = sum(1 for i in data if _extract_float(i.get("reference")) is not None and _extract_float(i.get("response")) is not None)
        print("Valid pairs (reference + response both parse to number):", valid, "/", len(data))
        if valid == 0 and len(data) > 0:
            print("Tip: If 'response' is long text, extract_float takes the first number. If model never outputs a number, all items are skipped.")

: 

: 

## 4. Install dependencies

In [ ]:
"""
Paste this into a Colab cell and run it. No requirements.txt needed.
Uses torch>=2.2 for Colab compatibility; skips triton and nvidia-* (Colab has its own CUDA).
"""
PACKAGES = [
    "addict==2.4.0",
    "affine==2.4.0",
    "aiohappyeyeballs==2.4.0",
    "aiohttp==3.10.5",
    "aiomultiprocess==0.9.0",
    "aiosignal==1.3.1",
    "annotated-types==0.7.0",
    "anyio==4.5.0",
    "asttokens==2.4.1",
    "async-timeout==4.0.3",
    "attrs==24.2.0",
    "beautifulsoup4==4.12.3",
    "bs4==0.0.2",
    "certifi==2024.8.30",
    "charset-normalizer==3.3.2",
    "click==8.1.7",
    "click-plugins==1.1.1",
    "cligj==0.7.2",
    "comm==0.2.2",
    "contextily==1.6.2",
    "contourpy==1.3.0",
    "coord-convert==0.2.1",
    "coverage==7.6.1",
    "crcmod==1.7",
    "cryptography==42.0.5",
    "cycler==0.12.1",
    "dashscope==1.20.10",
    "debugpy==1.8.5",
    "decorator==5.1.1",
    "distro==1.9.0",
    "dnspython==2.6.1",
    "exceptiongroup==1.2.2",
    "executing==2.1.0",
    "fastapi==0.115.0",
    "filelock==3.16.1",
    "fiona==1.10.1",
    "fonttools==4.53.1",
    "frozenlist==1.4.1",
    "fsspec==2024.9.0",
    "ftfy==6.2.3",
    "gast==0.5.4",
    "geographiclib==2.0",
    "geojson==3.1.0",
    "geopandas==0.14.3",
    "geopy==2.4.1",
    "grpcio==1.66.1",
    "grpcio-tools==1.62.3",
    "h11==0.14.0",
    "httpcore==1.0.5",
    "httpx==0.27.2",
    "huggingface-hub==0.25.0",
    "idna==3.10",
    "igraph==0.11.6",
    "importlib_metadata==7.1.0",
    "iniconfig==2.0.0",
    "ipykernel==6.29.5",
    "ipython==8.27.0",
    "jedi==0.19.1",
    "Jinja2==3.1.4",
    "jiter==0.5.0",
    "jmespath==0.10.0",
    "joblib==1.4.2",
    "json_repair==0.25.3",
    "jsonlines==4.0.0",
    "jupyter_client==8.6.3",
    "jupyter_core==5.7.2",
    "kiwisolver==1.4.7",
    "Levenshtein==0.25.1",
    "loguru==0.7.2",
    "lru-dict==1.3.0",
    "lxml==5.3.0",
    "MarkupSafe==2.1.5",
    "matplotlib==3.8.3",
    "matplotlib-inline==0.1.7",
    "mercantile==1.2.1",
    "modelscope==1.13.3",
    "mosstool==0.2.34",
    "mpmath==1.3.0",
    "multidict==6.1.0",
    "nest-asyncio==1.6.0",
    "networkx==3.3",
    "numpy==1.26.4",
    "open_clip_torch==2.26.1",
    "openai==1.46.0",
    "opencv-python==4.10.0.84",
    "packaging==24.1",
    "pandas==2.2.2",
    "parso==0.8.4",
    "path4gmns==0.9.8",
    "peft==0.12.0",
    "pexpect==4.9.0",
    "pillow==10.4.0",
    "platformdirs==4.3.6",
    "pluggy==1.5.0",
    "prompt_toolkit==3.0.47",
    "protobuf==4.25.5",
    "psutil==6.0.0",
    "ptyprocess==0.7.0",
    "pure_eval==0.2.3",
    "pyarrow==16.0.0",
    "pyarrow-hotfix==0.6",
    "pydantic==2.9.2",
    "pydantic_core==2.23.4",
    "pydeck==0.8.0",
    "Pygments==2.18.0",
    "pymongo==4.9.1",
    "pyparsing==3.1.4",
    "pyproj==3.6.1",
    "pytest==8.3.3",
    "pytest-cov==5.0.0",
    "pytest-cover==3.0.0",
    "pytest-coverage==0.0",
    "python-dateutil==2.9.0.post0",
    "python-moss==0.4.4",
    "pytz==2024.2",
    "PyYAML==6.0.2",
    "pyzmq==26.2.0",
    "rapidfuzz==3.9.7",
    "rasterio==1.3.9",
    "regex==2024.9.11",
    "requests==2.32.3",
    "safetensors==0.4.5",
    "scikit-learn==1.5.2",
    "scipy==1.13.0",
    "seaborn==0.13.2",
    "setproctitle==1.3.3",
    "shapely==2.0.6",
    "simplejson==3.19.2",
    "six==1.16.0",
    "sniffio==1.3.1",
    "snuggs==1.4.7",
    "sortedcontainers==2.4.0",
    "soupsieve==2.6",
    "sse-starlette==2.1.3",
    "stack-data==0.6.3",
    "starlette==0.38.5",
    "stringcase==1.2.0",
    "sympy==1.13.3",
    "tenacity==8.5.0",
    "texttable==1.7.0",
    "thefuzz==0.22.1",
    "threadpoolctl==3.5.0",
    "timm==1.0.9",
    "tokenizers==0.19.1",
    "tomli==2.0.1",
    "tomlkit==0.12.0",
    "torch>=2.2.0",
    "torchvision>=0.17.0",
    "tornado==6.4.1",
    "tqdm==4.66.5",
    "traitlets==5.14.3",
    "transformers==4.41.2",
    "trl==0.9.6",
    "types-protobuf==4.25.0.20240417",
    "typing_extensions==4.12.2",
    "tzdata==2024.1",
    "urllib3==2.2.3",
    "uvicorn==0.30.6",
    "wcwidth==0.2.13",
    "websocket-client==1.8.0",
    "xxhash==3.4.1",
    "xyzservices==2024.9.0",
    "yapf==0.40.2",
    "yarl==1.11.1",
    "zipp==3.18.1",
]

# Skip install if key deps already present (safe for Run All)
import subprocess
import sys
_skip = False
try:
    import torch
    import openai
    import google.generativeai
    _skip = True
except ImportError:
    pass
if _skip:
    print("Key deps already installed (skipping pip).")
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + PACKAGES
    subprocess.check_call(cmd)
    print("Requirements installed.")


: 

: 

In [ ]:
# Skip if gdal already available (safe for Run All)
try:
    import osgeo
    print("GDAL already installed.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdal"])


: 

: 

## 5. Set API key and run GDP baseline

In [ ]:
import os

# Section 3 sets CITYLENS_DATA_ROOT from Google Drive; only set fallback if not already set
os.environ.setdefault("CITYLENS_DATA_ROOT", "/content/drive/MyDrive/GeoBench/CityLens-Data")

# Paste your Gemini API key below (get free key at https://aistudio.google.com/apikey)
GOOGLE_API_KEY = ""  # paste key here as a string, e.g. "AIza..."
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
else:
    key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
    if not key:
        from getpass import getpass
        key = getpass("Paste GOOGLE_API_KEY: ")
    os.environ["GOOGLE_API_KEY"] = key
print("API key set (Gemini).")

: 

: 

In [ ]:
# Patch eval_task then run (ensures fallback applied every time)
import os
import subprocess
os.chdir("/content/CityLens")
base = "/content/CityLens-Data"
data_root = None
env_root = os.environ.get("CITYLENS_DATA_ROOT")
if env_root and os.path.isdir(env_root):
    for _b in ["Benchmark", "benchmark"]:
        if os.path.isdir(os.path.join(env_root, _b)):
            data_root = env_root
            break
drive_base = "/content/drive/MyDrive/GeoBench/CityLens-Data"
if data_root is None and os.path.isdir(drive_base):
    for _sub in ["CityLens-data", "CityLens-Data", "citylens-data", ""]:
        _cand = os.path.join(drive_base, _sub) if _sub else drive_base
        if os.path.isdir(_cand):
            _kids = [x.lower() for x in os.listdir(_cand)]
            if "dataset" in _kids and "benchmark" in _kids:
                data_root = _cand.rstrip(os.sep) or _cand
                break
if data_root is None and os.path.isdir(base):
    for dirpath, dirs, _ in os.walk(base):
        _d = [x.lower() for x in dirs]
        if "benchmark" in _d and "dataset" in _d:
            data_root = dirpath
            break
if data_root is None and os.path.isdir(base):
    for _sub in ["CityLens-data", "CityLens-Data", "citylens-data"]:
        _cand = os.path.join(base, _sub)
        if os.path.isdir(_cand):
            _kids = [x.lower() for x in os.listdir(_cand)]
            if "dataset" in _kids and "benchmark" in _kids:
                data_root = _cand
                break
if data_root is None and os.path.isdir("/content"):
    for dirpath, dirs, _ in os.walk("/content"):
        if dirpath.count(os.sep) > 6: break
        _d = [x.lower() for x in dirs]
        if "benchmark" in _d and "dataset" in _d:
            data_root = dirpath
            break
if data_root is None:
    _msg = "No Benchmark/ and Dataset/ found. "
    if os.path.isdir(base):
        _msg += "Under " + base + " saw: " + str(os.listdir(base)[:15]) + ". "
    else:
        _msg += base + " does not exist. "
    _msg += "Run section 3 (Google Drive + CityLens dataset) first to mount Drive and download or reuse data. Run the 'Verify dataset' cell to inspect paths."
    raise SystemExit(_msg)

# Skip if GDP eval results already exist (same model/prompt)
_results_file = os.path.join(data_root, "Results", "gdp_all_gemini-1.5-flash_simple.json")
if os.path.isfile(_results_file):
    os.environ["CITYLENS_DATA_ROOT"] = data_root
    print("CITYLENS_DATA_ROOT =", data_root)
    print("GDP eval already done (results exist). Skipping this cell.")
else:
    bench_gdp = os.path.join(data_root, "Benchmark", "gdp_all.json")
dataset_gdp = os.path.join(data_root, "Dataset", "all_global_gdp_task_all.json")
if not os.path.exists(bench_gdp) and os.path.exists(dataset_gdp):
    os.makedirs(os.path.dirname(bench_gdp), exist_ok=True)
    import shutil
    shutil.copy2(dataset_gdp, bench_gdp)
    print("Copied Dataset/all_global_gdp_task_all.json -> Benchmark/gdp_all.json")
elif not os.path.exists(bench_gdp):
    raise SystemExit("Neither Benchmark/gdp_all.json nor Dataset/all_global_gdp_task_all.json found under " + data_root)
bench_dir = os.path.join(data_root, "Benchmark")
url_csv = os.path.join(bench_dir, "image_urls.csv")
if not os.path.exists(url_csv):
    for r, _, files in os.walk(data_root):
        if "image_urls.csv" in files:
            import shutil
            shutil.copy2(os.path.join(r, "image_urls.csv"), url_csv)
            print("Copied image_urls.csv -> Benchmark/")
            break
if not os.path.exists(url_csv):
    gpath = "evaluate/global/global_indicator.py"
    with open(gpath) as f: gt = f.read()
    if "url_dict.get(image_name)" in gt and "file://" not in gt:
        gt = gt.replace("    url_file = url_file_path() if _USE_PATH_CONFIG else \"\"\n    df_url = pd.read_csv(url_file)\n    url_dict = dict(zip(df_url['image_name'], df_url['image_url']))", "    url_dict = {}\n    if _USE_PATH_CONFIG:\n        uf = url_file_path()\n        if os.path.exists(uf):\n            df_url = pd.read_csv(uf)\n            url_dict = dict(zip(df_url['image_name'], df_url['image_url']))")
        gt = gt.replace("        image_url = url_dict.get(image_name)\n        if not image_url:\n            raise ValueError", "        image_url = url_dict.get(image_name)\n        if not image_url:\n            _root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"\")\n            for cand in [os.path.join(_root, image_path), os.path.join(_root, \"satellite_image\", image_name), os.path.join(_root, \"street_view_image\", image_name)]:\n                if _root and os.path.isfile(cand):\n                    image_url = \"file://\" + os.path.abspath(cand)\n                    break\n        if not image_url:\n            raise ValueError(\"No URL or local file for \" + image_name)")
        with open(gpath, "w") as f: f.write(gt)
        print("Patched global_indicator for local image fallback.")
    if "file://" in gt and "os.walk(_root)" not in gt:
        with open(gpath) as f: gt = f.read()
        old = "                    break\n        if not image_url:\n            raise ValueError(\"No URL or local file for \" + image_name)"
        new = "                    break\n            if not image_url and _root:\n                for _dp, _, _fs in os.walk(_root):\n                    if image_name in _fs:\n                        image_url = \"file://\" + os.path.abspath(os.path.join(_dp, image_name))\n                        break\n                    if _dp[len(_root):].count(os.sep) >= 5: break\n        if not image_url:\n            raise ValueError(\"No URL or local file for \" + image_name)"
        if old in gt: gt = gt.replace(old, new); open(gpath, "w").write(gt); print("Added os.walk image search.")
    gt = open(gpath).read()
    if "No URL or local file" in gt and "return None  # skip missing image" not in gt:
        gt = gt.replace("raise ValueError(\"No URL or local file for \" + image_name)", "return None  # skip missing image")
        gt = gt.replace("        session = generate_session_map(city, d, task_name)\n    reference = d[\"reference\"]", "        session = generate_session_map(city, d, task_name)\n    if session is None: return None\n    reference = d[\"reference\"]")
        open(gpath, "w").write(gt)
        print("Patched: skip tasks with missing images.")
    ga = "evaluate/gemini_adapter.py"
    if os.path.exists(ga):
        with open(ga) as f: gat = f.read()
        if "file://" not in gat and "url.startswith" not in gat:
            old = "            try:\n                resp = requests.get(url, timeout=30)"
            new = "            if url.startswith(\"file://\"):\n                path = url[7:].replace(\"\\\", os.sep)\n                with open(path, \"rb\") as fp:\n                    b64 = base64.b64encode(fp.read()).decode(\"utf-8\")\n                mime = \"image/jpeg\"\n                if path.lower().endswith(\".png\"): mime = \"image/png\"\n                parts.append({\"inline_data\": {\"mime_type\": mime, \"data\": b64}})\n                continue\n            try:\n                resp = requests.get(url, timeout=30)"
            if old in gat:
                gat = gat.replace(old, new)
                with open(ga, "w") as f: f.write(gat)
                print("Patched gemini_adapter for file:// URLs.")
os.environ["CITYLENS_DATA_ROOT"] = data_root
print("CITYLENS_DATA_ROOT =", data_root)
path = "/content/CityLens/evaluate/global/global_indicator.py"
with open(path) as f:
    t = f.read()
needs_fallback = "if not task_path or not response_path" not in t
needs_guard = "if output_dir and not os.path.exists(output_dir)" not in t and "os.makedirs(output_dir, exist_ok=True)" in t
needle = None
if needs_fallback:
    import re
    m = re.search(r"^\s*output_dir = os\.path\.dirname\(response_path\)", t, re.MULTILINE)
    if m: needle = m.group(0)
    if not needle and "output_dir = os.path.dirname(response_path)" in t:
        needle = "output_dir = os.path.dirname(response_path)"
    if needle:
        insert = "    # Fallback when path_config not loaded (Colab)\n    if not task_path or not response_path:\n        data_root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"/content/CityLens-Data\")\n        task_path = os.path.join(data_root, \"Benchmark\", f\"{task_name}_{city}.json\")\n        response_path = os.path.join(data_root, \"Results\", f\"{task_name}_{city}_{model_name_full}_{prompt_type}.json\")\n    "
        before, _, after = t.partition(needle)
        t = before + insert + needle + after
        print("Patched global_indicator.py (fallback injected).")
    else:
        print("Warning: could not find output_dir line to patch fallback.")
if needs_guard:
    import re
    t = t.replace(" if not os.path.exists(output_dir):", " if output_dir and not os.path.exists(output_dir):")
    t = t.replace("    if not os.path.exists(output_dir):", "    if output_dir and not os.path.exists(output_dir):")
    t = re.sub(r"(\n)(\s+)os\.makedirs\(output_dir, exist_ok=True\)", r"\1\2if output_dir and not os.path.exists(output_dir):\n\2    os.makedirs(output_dir, exist_ok=True)", t)
    print("Patched makedirs guard (no makedirs on empty output_dir).")
if (needs_fallback and needle) or needs_guard:
    with open(path, "w") as f: f.write(t)
i = t.find("def eval_task(city, model_name, num_process, prompt_type, task_name):")
j = t.find("\ndef main(", i)
try:
    with open('colab_eval_task_replacement.txt') as f: REPL = f.read()
except FileNotFoundError:
    REPL = '''def eval_task(city, model_name, num_process, prompt_type, task_name):
    model_name_full = model_name.replace("/", "_")
    data_root = os.environ.get("CITYLENS_DATA_ROOT", "/content/CityLens-Data")
    task_path = os.path.join(data_root, "Benchmark", f"{task_name}_{city}.json")
    response_path = os.path.join(data_root, "Results", f"{task_name}_{city}_{model_name_full}_{prompt_type}.json")
    output_dir = os.path.dirname(response_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    with open(task_path, "r") as f:
        data = json.load(f)
    if len(data) > GLOBAL_TASK_NUM:
        data = random.sample(data, GLOBAL_TASK_NUM)
    args_list = [(city, model_name, d, prompt_type, task_name) for d in data]
    response = []
    with Pool(num_process) as pool:
        print("Processing tasks in parallel...")
        for result in tqdm(pool.imap(single_eval_task, args_list), total=len(data)):
            if result:
                response.append(result)
    with open(response_path, "w") as f:
        json.dump(response, f, indent=4, ensure_ascii=False)
'''
if i != -1 and j != -1:
    t = t[:i] + REPL + chr(10) + chr(10) + t[j:]
    with open(path, 'w') as f: f.write(t)
    print('Replaced eval_task.')
env = {**os.environ, "CITYLENS_DATA_ROOT": data_root}
r = subprocess.run(["python", "-B", "-m", "evaluate.global.global_indicator", "--city_name=all", "--mode=eval", "--model_name=gemini-1.5-flash", "--prompt_type=simple", "--num_process=10", "--task_name=gdp"], cwd="/content/CityLens", env=env, capture_output=True, text=True)
if r.stdout: print(r.stdout)
if r.stderr: print(r.stderr)
r.check_returncode()


: 

: 

In [ ]:
# Run metrics (inline: load JSON + compute here, then patch metrics.py)
import os
print('>>> METRICS CELL v2 - if you see this, you are running the updated code')
base = "/content/CityLens-Data"
drive_base = "/content/drive/MyDrive/GeoBench/CityLens-Data"
data_root = None
results_file = "Results/gdp_all_gemini-1.5-flash_simple.json"
env_root = os.environ.get("CITYLENS_DATA_ROOT")
if env_root and os.path.isfile(os.path.join(env_root, results_file)):
    data_root = env_root
if data_root is None and os.path.isdir(drive_base):
    for dirpath, dirs, _ in os.walk(drive_base):
        cand = os.path.join(dirpath, results_file)
        if os.path.isfile(cand):
            data_root = dirpath
            break
if data_root is None and os.path.isdir(base):
    for dirpath, dirs, _ in os.walk(base):
        cand = os.path.join(dirpath, results_file)
        if os.path.isfile(cand):
            data_root = dirpath
            break
if data_root is None:
    data_root = os.environ.get("CITYLENS_DATA_ROOT", "/content/CityLens-Data/CityLens-data")
os.environ["CITYLENS_DATA_ROOT"] = data_root
print("CITYLENS_DATA_ROOT =", data_root)
_resp = os.path.join(data_root, 'Results', 'gdp_all_gemini-1.5-flash_simple.json')
_summ = os.path.join(data_root, 'Results', 'summary.csv')
import json as _json
import re as _re
import csv as _csv
try:
    _jd = _json.load(open(_resp))
except FileNotFoundError:
    print('ERROR: Results file not found:', _resp); _jd = []
except Exception as _e:
    print('ERROR loading JSON:', _e); _jd = []
print('Results JSON len:', len(_jd))
if len(_jd) == 0:
    print('ERROR: Results file is EMPTY. Re-run the GDP eval cell (Patch eval_task then run) and wait for it to finish.')
def _extract_float(v):
    if v is None: return None
    if isinstance(v, (float, int)): return float(v)
    if isinstance(v, str):
        s = v.replace(',', '').replace('$', '').replace(chr(163), '').strip()
        if len(s) <= 100:
            try: return float(s)
            except ValueError: pass
        m = _re.search(r'-?[\\d,]+\\.?\\d*', v)
        if m: return float(m.group().replace(',', ''))
    return None
y_true, y_pred = [], []
for _it in _jd:
    ref = _extract_float(_it.get('reference'))
    rsp = _it.get('response')
    if rsp is not None and not isinstance(rsp, str): rsp = str(rsp)
    pred = _extract_float(rsp)
    if ref is not None and pred is not None:
        y_true.append(ref); y_pred.append(pred)
print('length of vals:', len(y_true), len(y_pred))
if len(y_true) == 0 and _jd:
    _it = _jd[0]
    print('DEBUG first item keys:', list(_it.keys()))
    print('  reference', type(_it.get('reference')).__name__, repr(str(_it.get('reference'))[:150]))
    print('  response', type(_it.get('response')).__name__, repr(str(_it.get('response'))[:150]))
    print('  _extract_float(ref)=', _extract_float(_it.get('reference')), '_extract_float(resp)=', _extract_float(_it.get('response')))
if len(y_true) > 0:
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    mse, mae, r2 = mean_squared_error(y_true, y_pred), mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    print('MSE', mse, 'MAE', mae, 'R2', r2, 'RMSE', rmse)
    os.makedirs(os.path.dirname(_summ), exist_ok=True)
    _write_header = not os.path.exists(_summ)
    with open(_summ, 'a', newline='') as _f:
        _w = _csv.DictWriter(_f, fieldnames=['city','model','prompt_type','MSE','MAE','R2','RMSE'])
        if _write_header: _w.writeheader()
        _w.writerow({'city':'all','model':'gemini-1.5-flash','prompt_type':'simple','MSE':mse,'MAE':mae,'R2':r2,'RMSE':rmse})
    print('Metrics written to', _summ)
else:
    print('No valid data found.')
print('Metrics done.')
path = '/content/CityLens/evaluate/global/metrics.py'
with open(path) as f:
    t = f.read()
import re as _re
if 'match = re.search' not in t and 'def extract_float' in t:
    _start = t.find('def extract_float(value):')
    _end = t.find('def process_json(', _start)
    if _start != -1 and _end != -1:
        _new_ef = '''def extract_float(value):
    if isinstance(value, (float, int)):
        return float(value)
    if isinstance(value, str):
        value_str = value.replace(\",\", \"\").replace(\"$\", \"\").replace(\"£\", \"\").strip()
        if len(value_str) <= 100:
            try:
                return float(value_str)
            except ValueError:
                pass
        match = re.search(r\"-?[\\d,]+\\.?\\d*\", value)
        if match:
            try:
                return float(match.group().replace(\",\", \"\"))
            except ValueError:
                pass
    return None

'''
        t = t[:_start] + _new_ef + t[_end:]
        print('Patched extract_float for long/verbose responses.')
    else:
        pass
start = t.find('args = parser.parse_args()')
end_line = t.find('y_true, y_pred = process_json(response_path, args.prompt_type)')
if start != -1 and end_line != -1:
    line_start = t.rfind(chr(10), 0, start) + 1
    indent = t[line_start:start]
    if indent.strip() or len(indent) > 12:
        indent = '    '
    METRICS_MAIN = indent + "args = parser.parse_args()\n" + indent + "model_name_full = args.model_name.replace(\"/\", \"_\")\n" + indent + "data_root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"/content/CityLens-Data/CityLens-data\")\n" + indent + "response_path = os.path.join(data_root, \"Results\", f\"{args.task_name}_{args.city_name}_{model_name_full}_{args.prompt_type}.json\")\n" + indent + "summary_path = os.path.join(data_root, \"Results\", \"summary.csv\")\n" + indent + "os.makedirs(os.path.dirname(summary_path), exist_ok=True)\n" + indent + "y_true, y_pred = process_json(response_path, args.prompt_type)\n"
    end = t.find(chr(10), end_line)
    end = (end + 1) if end != -1 else len(t)
    t = t[:line_start] + METRICS_MAIN + t[end:]
    with open(path, 'w') as f:
        f.write(t)
    print('Replaced metrics path block (env-based).')


: 

: 

## 6. (Optional) Copy results to Drive

```python
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/CityLens-Data/Results /content/drive/MyDrive/
```